# Experiment 3: ensemble_v1

**Hypothesis:** Whisper Small and Whisper Large-V3 make different mistakes. Selecting the better transcript per sample improves overall quality.

**Input:** `submission_beam_small.csv` + `submission_large_v3.csv`

**Method:** Per-sample heuristic selection. Reject empty, single-char, punctuation-only, high-repetition, and hallucinated-script predictions. Prefer Large-V3 unless Small clearly produces a better transcript.

**Output:** `submissions/submission_ensemble.csv`

## 1. Setup

In [ ]:
import csv, re, unicodedata
from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path(".").resolve().parent
SUBMISSION_DIR = PROJECT_ROOT / "submissions"

small_path = SUBMISSION_DIR / "submission_beam_small.csv"
large_path = SUBMISSION_DIR / "submission_large_v3.csv"
ensemble_path = SUBMISSION_DIR / "submission_ensemble.csv"
sample_path = PROJECT_ROOT / "SampleSubmission.csv"

assert small_path.exists(), f"Missing {small_path} — run Experiment 1 first"
assert large_path.exists(), f"Missing {large_path} — run Experiment 2 first"
print("Both submission files found.")

## 2. Load Predictions

In [ ]:
def load_submission(path):
    preds = {}
    with open(path, "r", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            preds[row["ID"]] = row["Target"]
    return preds

small_preds = load_submission(small_path)
large_preds = load_submission(large_path)

# Load test IDs in order
test_ids = []
with open(PROJECT_ROOT / "Test.csv", "r", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        test_ids.append(row["ID"])

print(f"Small predictions: {len(small_preds)}")
print(f"Large predictions: {len(large_preds)}")
print(f"Test IDs: {len(test_ids)}")

## 3. Quality Scoring

In [ ]:
GARBLED_SCRIPTS = {
    "KHMER", "DEVANAGARI", "THAI", "TIBETAN", "MYANMAR",
    "BENGALI", "GUJARATI", "KANNADA", "TAMIL", "TELUGU",
    "MALAYALAM", "SINHALA", "LAO", "GEORGIAN", "ARMENIAN",
    "ETHIOPIC", "CHEROKEE", "CANADIAN",
}

def is_empty(text):
    """Empty or single character."""
    return not text or len(text.strip()) <= 1

def is_punct_only(text):
    """Only punctuation/whitespace, no alphanumeric."""
    return bool(text) and all(not c.isalnum() for c in text)

def is_garbled(text):
    """Contains hallucinated non-Latin scripts (Khmer, Devanagari, etc.)."""
    if not text or len(text) <= 2:
        return False
    garbled = 0
    for ch in text:
        try:
            script = unicodedata.name(ch, "").split()[0]
            if script in GARBLED_SCRIPTS:
                garbled += 1
        except (ValueError, IndexError):
            pass
    return garbled / len(text) > 0.3

def is_repetitive(text):
    """More than 50% of tokens are the same word."""
    if not text:
        return False
    words = text.split()
    if len(words) <= 3:
        return False
    counts = Counter(words)
    return counts.most_common(1)[0][1] > len(words) * 0.5

def quality_score(text):
    """Score 0-100. Higher = better quality."""
    if is_empty(text):
        return 0
    if is_punct_only(text):
        return 2
    if is_garbled(text):
        return 5
    if is_repetitive(text):
        return 10
    
    score = 50
    length = len(text)
    words = text.split()
    
    # Length bonus
    if 20 <= length <= 500:
        score += 20
    elif 10 <= length <= 1000:
        score += 10
    
    # Word count bonus
    if len(words) >= 3:
        score += 10
    
    # Unique word ratio bonus
    unique_ratio = len(set(words)) / max(len(words), 1)
    score += int(unique_ratio * 20)
    
    # Penalize very high non-ASCII ratio
    non_ascii = sum(1 for c in text if ord(c) > 127)
    if non_ascii / max(len(text), 1) > 0.5:
        score -= 15
    
    return min(score, 100)

# Verify scoring works
print("Scoring tests:")
for label, text in [
    ("Empty", ""),
    ("Dot", "."),
    ("Repetitive", "kwa " * 20),
    ("Good text", "Amaato abali gali ku mazzi amateefu"),
]:
    print(f"  {label:12s} -> score={quality_score(text)}")

## 4. Ensemble Selection

In [ ]:
LARGE_PREFERENCE_MARGIN = 10  # Small must beat Large by this margin to win

ensemble_preds = {}
choice_stats = {"small": 0, "large": 0, "tie_large": 0}
lang_choices = {}

for tid in test_ids:
    lang = tid.split("_")[0]
    lang_choices.setdefault(lang, {"small": 0, "large": 0, "tie": 0})
    
    s_text = small_preds.get(tid, "")
    l_text = large_preds.get(tid, "")
    
    s_score = quality_score(s_text)
    l_score = quality_score(l_text)
    
    # Small must win by LARGE_PREFERENCE_MARGIN to be chosen
    if s_score > l_score + LARGE_PREFERENCE_MARGIN:
        ensemble_preds[tid] = s_text
        choice_stats["small"] += 1
        lang_choices[lang]["small"] += 1
    elif l_score > s_score:
        ensemble_preds[tid] = l_text
        choice_stats["large"] += 1
        lang_choices[lang]["large"] += 1
    else:
        # Tie or marginal small advantage -> prefer large
        ensemble_preds[tid] = l_text
        choice_stats["tie_large"] += 1
        lang_choices[lang]["tie"] += 1

print("Ensemble selection:")
print(f"  Large-V3 wins:      {choice_stats['large']}")
print(f"  Small wins:         {choice_stats['small']}")
print(f"  Ties (prefer large): {choice_stats['tie_large']}")
print()

print("Per-language:")
for lang in sorted(lang_choices):
    lc = lang_choices[lang]
    total = lc['small'] + lc['large'] + lc['tie']
    print(f"  {lang}: small={lc['small']}, large={lc['large']}, tie={lc['tie']} (n={total})")

## 5. Write Ensemble Submission

In [ ]:
with open(ensemble_path, "w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["ID", "Target"])
    for tid in test_ids:
        writer.writerow([tid, ensemble_preds.get(tid, "")])

print(f"Ensemble submission written to: {ensemble_path}")
print(f"Total predictions: {len(ensemble_preds)}")

# Validate
if sample_path.exists():
    with open(sample_path, "r", encoding="utf-8") as f:
        expected = {row["ID"] for row in csv.DictReader(f)}
    with open(ensemble_path, "r", encoding="utf-8") as f:
        submitted = {row["ID"] for row in csv.DictReader(f)}
    missing = expected - submitted
    empty = sum(1 for tid in test_ids if not ensemble_preds.get(tid, "").strip())
    if missing:
        print(f"VALIDATION FAILED: Missing {len(missing)} IDs!")
    elif empty:
        print(f"WARNING: {empty} IDs have empty transcriptions")
    else:
        print("VALIDATION PASSED")

## 6. Comparative Quality Analysis

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))
from scripts.quality_analysis import analyze, is_garbled, is_repetitive

# Generate ensemble report
report = analyze(str(ensemble_path), "ensemble_v1")
report_path = PROJECT_ROOT / "reports" / "quality_report_ensemble_v1.md"
report_path.parent.mkdir(exist_ok=True)
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

# Side-by-side comparison
print("=" * 70)
print("COMPARATIVE ANALYSIS: Small vs Large-V3 vs Ensemble")
print("=" * 70)
print()
print(f"{'Metric':<25} {'Small':>10} {'Large-V3':>10} {'Ensemble':>10}")
print("-" * 55)

for label, preds in [("Small", small_preds), ("Large-V3", large_preds), ("Ensemble", ensemble_preds)]:
    vals = [preds.get(tid, "") for tid in test_ids]
    empty_c = sum(1 for v in vals if not v.strip())
    garbled_c = sum(1 for v in vals if is_garbled(v))
    rep_c = sum(1 for v in vals if is_repetitive(v))
    non_empty = [v for v in vals if v.strip()]
    avg_len = sum(len(v) for v in non_empty) / max(len(non_empty), 1)
    unique_c = len(set(non_empty))
    dup_c = len(non_empty) - unique_c
    globals()[f"{label.lower().replace('-','_')}_stats"] = {
        "empty": empty_c, "garbled": garbled_c, "rep": rep_c,
        "avg_len": avg_len, "unique": unique_c, "dup": dup_c,
    }

# Print comparison table
for metric in ["empty", "garbled", "rep", "dup", "avg_len", "unique"]:
    s = globals()["small_stats"][metric]
    l = globals()["large_v3_stats"][metric]
    e = globals()["ensemble_stats"][metric]
    fmt = ".0f" if metric == "avg_len" else "d"
    print(f"{metric:<25} {s:>10{fmt}} {l:>10{fmt}} {e:>10{fmt}}")

print()
print(report)